# Project Setup — Stock Market Pipeline
This notebook sets up the Unity Catalog infrastructure for the stock market pipeline.

**What gets created:**
- External Location — registers the ADLS Gen2 storage path with Databricks
- Catalog — top level container for all pipeline data
- Schemas — bronze, silver, gold layers
- Volume — easy file reference path for the bronze layer

## Step 1 — Create External Location
Registers the Azure Data Lake Storage path with Unity Catalog using the storage credential.
This allows Databricks to read and write to the marketdata container.

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS stock_pipeline_ext_location
  URL 'abfss://marketdata@stockpipelineadls.dfs.core.windows.net/'
  WITH (STORAGE CREDENTIAL `stock_pipeline_ext_sc`)
  COMMENT 'External Location for Stock Market Pipeline'

## Step 2 — Create Catalog
Creates the top level Unity Catalog container pointing to the marketdata storage container.

In [0]:
CREATE CATALOG IF NOT EXISTS stock_pipeline
MANAGED LOCATION 'abfss://marketdata@stockpipelineadls.dfs.core.windows.net/'
COMMENT 'Stock Market Pipeline Data Lakehouse';

## Step 3 — Create Schemas
Creates the three medallion architecture layers inside the catalog.

In [0]:
USE CATALOG stock_pipeline;

CREATE SCHEMA IF NOT EXISTS bronze
    COMMENT 'Raw ingested stock data';

CREATE SCHEMA IF NOT EXISTS silver
    COMMENT 'Cleaned and validated stock data';

CREATE SCHEMA IF NOT EXISTS gold
    COMMENT 'Aggregated business metrics';

SHOW SCHEMAS;

## Step 4 — Create Volume
Creates an external volume pointing to the bronze stocks folder.
Allows referencing files as /Volumes/stock_pipeline/bronze/stock_data/ instead of the full abfss:// path.

In [0]:
USE CATALOG stock_pipeline;
USE SCHEMA bronze;

CREATE EXTERNAL VOLUME IF NOT EXISTS stock_pipeline.bronze.stock_data
    LOCATION 'abfss://marketdata@stockpipelineadls.dfs.core.windows.net/bronze/stocks'
    COMMENT 'Bronze layer raw stock data';

SHOW VOLUMES;

## Step 5 — Verify Setup
Confirm the volume is working and all 37 ticker folders are accessible.

In [0]:
LIST '/Volumes/stock_pipeline/bronze/stock_data/'